# bug-report-checker: fine-tune Laya on Kaggle

Settings: **Accelerator GPU T4**, **Internet On**. Add a private dataset with `train_v2.jsonl`, `train_v3.jsonl`, `dev_v3.jsonl` and `eval_v2.jsonl` (all from `python -m bug_report_checker.dataset.laya_format`).

Training is `laya.train.finetune` (what `laya-train` runs) from the pinned `laya` package: the published notebook's loop and RLCD objective, one GPU. laya 0.4.0 evaluates the base model before moving it to the GPU and fails on CUDA, with or without `--eval`, so the checkpoint is moved right after loading.

Run 2 (`train_v2`, Sonnet labels) and run 3 (`train_v3`, Opus labels and the ordinary top-up) are trained in the same session, and base, run 2 and run 3 predict dev v3 and eval v2 on the same GPU, so the comparison does not mix CPU fp32 with GPU fp16 (D20). Runs are chosen on dev v3; eval v2 predictions are written too but scored only after the choice is recorded (D18).

Download from the notebook output: `preds/` and the checkpoint zips.

In [ ]:
!pip install -q laya==0.4.0

In [ ]:
import glob
import os

from huggingface_hub import snapshot_download


def find(name):
    # Kaggle's mount path for an input dataset varies; find the file, don't guess.
    [path] = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return path


FILES = {n: find(f"{n}.jsonl") for n in ("train_v2", "train_v3", "dev_v3", "eval_v2")}
print(FILES)
WORK = "/kaggle/working"
# The base checkpoint at the revision the local dry-run and baseline used.
REVISION = "7b928d828b7b0e022f929d9bd2e44165aa270148"
BASE = os.path.join(
    snapshot_download(
        "convaiinnovations/laya", revision=REVISION, allow_patterns="typed-decisions/*"
    ),
    "typed-decisions",
)
print(BASE)

In [ ]:
import laya.train as lt

_load_checkpoint = lt.load_checkpoint


def load_on_device(model_dir):
    # laya 0.4.0 runs the before-training eval while the model is still on the CPU.
    model, tok, cfg = _load_checkpoint(model_dir)
    return model.to(lt.resolve_device("auto")), tok, cfg


lt.load_checkpoint = load_on_device
MODELS = {"base": BASE}
for run, data in (("run2", "train_v2"), ("run3", "train_v3")):
    MODELS[run] = f"{WORK}/bug-report-checker-{run}"
    summary = lt.finetune(FILES[data], BASE, MODELS[run], lt.TrainConfig(seed=0))
    print(run, summary)

In [ ]:
import json

import laya

# P(true) of every noul question per report; long reports are scanned in windows.
os.makedirs(f"{WORK}/preds", exist_ok=True)
for name, path in MODELS.items():
    agent = laya.load(path, device="cuda")
    for split in ("dev_v3", "eval_v2"):
        with open(FILES[split], encoding="utf-8") as f:
            rows = [json.loads(line) for line in f]
        with open(f"{WORK}/preds/{name}_{split}.jsonl", "w", encoding="utf-8") as out:
            for r in rows:
                answers = agent.predict_long(r["state"], r["questions"])["answers"]
                p_true = {q: a["noul"] for q, a in answers.items()}
                out.write(json.dumps({"id": r["id"], **p_true}) + "\n")
        print(name, split, len(rows))
    del agent

In [ ]:
%cd /kaggle/working
!zip -qr preds.zip preds
!zip -qr bug-report-checker-run2.zip bug-report-checker-run2 -x '*/checkpoint_latest/*'
!zip -qr bug-report-checker-run3.zip bug-report-checker-run3 -x '*/checkpoint_latest/*'
!ls -lh